In [ ]:
# !pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

Some common tools built in with the community 

https://docs.langchain.com/oss/python/integrations/tools

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.tools import RequestsGetTool
from langchain_community.tools import GmailSendMessage
from langchain_community.tools import QuerySQLDataBaseTool


## Built-in Tool - DuckDuckGo Search

Tools are runnable so they have the invoke function

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in Nepal today')

print(results)

The Kathmandu Post : Find the latest breaking news from Nepal, opinion & analysis on Nepali politics, business, culture & arts, sports, movies, food & travel, books, education, auto & more at kathmandupost.com. Agni Sanchar Pvt.Ltd. Registration Number in Company Register Office Nepal : 371624/82/83 Department Of Information (Suchana Bibhag) Registration Number : 5245-2082/2083 Email : agnipost2025@gmail.com … ...ещё. Stay informed with top world news today. The Associated Press aims to keep you up-to-date with breaking world news stories around the globe. Nepal News is Nepal’s first and #1 online news portal. Get breaking news, politics, business, culture, sports, entertainment, analysis, and Nepal facts. A list of Nepal Nepali, and English newspapers featuring business, politics, sports, entertainment, travel, jobs, education, lifestyles, and more published in Nepal, Kathmandu and other regions.


In [ ]:
# META DATA OF TOOLS
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


## Built-in Tool - Shell Tool

In [10]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

# results = shell_tool.invoke('whoami')
results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
tools.md
tools_in_langchain.ipynb



## Custom Tools

https://docs.langchain.com/oss/python/langchain/tools

In [11]:
from langchain_core.tools import tool

This docstring is not absolutely neccessary for running But it is recommended as the llm will know what this tool/function do through this docstring 

In [12]:
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [13]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [14]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [15]:
result = multiply.invoke({"a":3, "b":5})

In [16]:
print(result)

15


In [17]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [18]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


llm gets this schema in proper json format 

## Method 2 - Using StructuredTool

In [20]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [22]:
class MultiplyInput(BaseModel):
    # a: int = Field(required=True, description="The first number to add")
    a: int = Field( description="The first number to add")
    # b: int = Field(required=True, description="The second number to add")
    b: int = Field( description="The second number to add")

means that required=True is no longer supported.

In Pydantic v2, the field is already required if:

it has no default value, or
you use ... (ellipsis).

In [23]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [ ]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput  # this is important pydantic class applies the schema
)

In [25]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class

In [26]:
from langchain.tools import BaseTool
from typing import Type

In [28]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number to add")
    b: int = Field(..., description="The second number to add")

In [ ]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int: # this _run is the function where logic must be written
        return a * b

In [30]:
multiply_tool = MultiplyTool()

In [31]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Toolkit

In [32]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [33]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [35]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)
    result=tool.invoke({'a':12,'b':12})
    print(result)
    


add => Add two numbers
24
multiply => Multiply two numbers
144


**Short answer: Usually, no.**

LangChain **Tool** objects are not regular Python functions. They have a standardized interface, so `.invoke()` generally expects **one input object** (often a dictionary or a string), not multiple positional arguments.

For example, if your tool was created with:

```python
class MultiplyInput(BaseModel):
    a: int
    b: int
```

then the correct way is:

```python
result = tool.invoke({"a": 12, "b": 12})
```

---

## Why can't I do this?

```python
tool.invoke(12, 12)   # ❌
```

Because the signature of `invoke()` is essentially:

```python
invoke(input, config=None)
```

It accepts **one input argument**, not `a` and `b` separately.

---

## Can I do this?

```python
tool(12, 12)
```

No. A `Tool` object is not a normal Python function.

---

## If you want normal function syntax

Keep the original function and call it directly.

```python
def multiply(a: int, b: int):
    return a * b

# Normal function call
print(multiply(12, 12))
```

When you wrap it as a tool:

```python
tool = StructuredTool.from_function(...)
```

you use:

```python
tool.invoke({"a": 12, "b": 12})
```

---

## For tools with a single input

If a tool has only one argument:

```python
@tool
def square(x: int):
    return x * x
```

you can often write:

```python
tool.invoke({"x": 5})
```

Some tools also allow:

```python
tool.invoke(5)
```

because there's only one parameter. This does **not** work for tools with multiple parameters.

---

## Summary

| Call                              | Works?                  |
| --------------------------------- | ----------------------- |
| `multiply(12, 12)`                | ✅ Yes (normal function) |
| `tool.invoke({"a": 12, "b": 12})` | ✅ Yes (LangChain tool)  |
| `tool.invoke(12, 12)`             | ❌ No                    |
| `tool(12, 12)`                    | ❌ No                    |

The reason is that LangChain tools are designed so that **agents and models can call them automatically**. Passing inputs as a structured object (dictionary or schema) makes it possible for the framework to validate inputs, serialize them, and integrate with different models consistently.
